# Dataset Publication: Archiving to Hugging Face Hub

## Overview
This notebook handles the final step of the data engineering pipeline: publishing the processed, leakage-free, bidirectional dataset to the Hugging Face Hub. By hosting the data remotely, we ensure research reproducibility and simplify the data loading process for subsequent training notebooks using the `datasets` library.

## Key Actions
1. **Authentication:** Securely logs into the Hugging Face Hub using the `HF_TOKEN` stored in Colab Secrets.
2. **Repository Management:** Automatically creates the dataset repository (`abhinandansamal/odia-german-parallel-corpus-research`) if it does not already exist.
3. **Artifact Upload:**
  * **Research Splits:** Uploads the specific `train`, `validation`, and `test` JSONL files to a `data/` subdirectory to facilitate automatic splitting detection.
  * **Raw Corpus:** Archives the `authentic_corpus_final.jsonl` for full transparency of the source data.

## Requirements
* **Libraries:** `huggingface_hub`
* **Prerequisites:** A valid Hugging Face Write Token and the processed JSONL files from the "Data Preparation" step.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Install the library to interact with the Hub
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login, HfApi
import os
from google.colab import userdata

In [ ]:
# 1. Authenticate with Hugging Face
huggingface_token = userdata.get('HF_TOKEN')
login(token=huggingface_token)

In [ ]:
# --- Configuration ---
HF_USERNAME = "abhinandansamal"
DATASET_REPO_ID = f"{HF_USERNAME}/odia-german-parallel-corpus-research"

# Local directories and file paths
DATA_DIR = "/content/drive/MyDrive/Research_Paper_Publication/data/transformed/"

# The specific split files created by your 'process_data' function
SPLIT_FILES = {
    "train": os.path.join(DATA_DIR, "train_bidirectional.jsonl"),
    "validation": os.path.join(DATA_DIR, "val_bidirectional.jsonl"),
    "test": os.path.join(DATA_DIR, "test_bidirectional.jsonl")
}

# The main corpora
AUTHENTIC_CORPUS = os.path.join(DATA_DIR, "authentic_corpus_final.jsonl")

In [ ]:
# --- Initialize HuggingFace API ---
api = HfApi()

print(f"🚀 Ensuring dataset repository exists: {DATASET_REPO_ID}")
api.create_repo(
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    exist_ok=True
)

🚀 Ensuring dataset repository exists: abhinandansamal/odia-german-parallel-corpus-research


RepoUrl('https://huggingface.co/datasets/abhinandansamal/odia-german-parallel-corpus-research', endpoint='https://huggingface.co', repo_type='dataset', repo_id='abhinandansamal/odia-german-parallel-corpus-research')

In [ ]:
# 2. Upload Split Files (Critical for Research Reproducibility)
# We upload these so that 'load_dataset()' can find them automatically
print("\n📦 Uploading Research Splits...")
for split_name, local_path in SPLIT_FILES.items():
    if os.path.exists(local_path):
        filename = os.path.basename(local_path)
        print(f"Uploading {split_name} split: {filename}...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=f"data/{filename}", # Storing in a 'data' subfolder is standard
            repo_id=DATASET_REPO_ID,
            repo_type="dataset",
        )
    else:
        print(f"⚠️ Warning: {local_path} not found. Skipping...")


📦 Uploading Research Splits...
Uploading train split: train_bidirectional.jsonl...
Uploading validation split: val_bidirectional.jsonl...
Uploading test split: test_bidirectional.jsonl...


In [ ]:
# 3. Upload the Full Authentic Corpus
if os.path.exists(AUTHENTIC_CORPUS):
    print(f"\n📄 Uploading Full Authentic Corpus...")
    api.upload_file(
        path_or_fileobj=AUTHENTIC_CORPUS,
        path_in_repo="authentic_corpus_final.jsonl",
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
    )

print("\n✅ Success! Dataset and splits uploaded.")
print(f"🔗 View Repository: https://huggingface.co/datasets/{DATASET_REPO_ID}")


📄 Uploading Full Authentic Corpus...

✅ Success! Dataset and splits uploaded.
🔗 View Repository: https://huggingface.co/datasets/abhinandansamal/odia-german-parallel-corpus-research
